<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma3:12b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

## 2 - Adding **`Memory`**

In the previous chapter, we covered which LLM to choose using various inference engines. In this chapter we will cover how to give it memory:

![../images/ch4.png](../images/ch4.png)

The issue with the `TinyAgent` that we have thus far, is that it does not track and remember its previous conversations, it is stateless. Let us demonstrate with an example by using the Agent from Chapter 2:

In [2]:
from rich import print
from illustrated_agents.chapters.ch2 import TinyAgent

ch_2_agent = TinyAgent(llm=llm)
response = ch_2_agent.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Hi Maarten and Jay! It's fantastic to hear from you! "An Illustrated Guide to AI Agents" is getting a *lot* of buzz
and is considered a really valuable resource in the AI space. I've heard excellent things about it. 

It's great to connect with the authors directly.  What can I do for you? Are you:

* **Promoting the book?** (I can help spread the word!)
* **Looking for feedback?** (I and many others would love to share our thoughts.)
* **Just saying hello?** (Welcome!) 



I'm excited to chat with you both.  Congratulations on your excellent work!

When we query the model again asking whether it knows our names, it seems to have forgotten them! In fact, it actually hasn't forgotten our name but instead has never received that information. Everytime you query an LLM it starts from an blank slate, one you have to fill yourself. So without telling the model the our conversation history, it has no way of knowing.

In [5]:
response = ch_2_agent.run("Hi! What are our names?")
print(response)

My name is Gemma 4. I was developed by Google DeepMind.

I do not know what your name is, as you have not told me!


Although the trajectory does show that we gave it our names first, the Agent had no access to this trajectory information:

In [6]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(ch_2_agent.trajectory)

## 3 - The **`Memory`** Module

As covered in the book, there are many ways to build up memory which can be quite difficult. In this example, we are going to keep it simple and only track the conversation history. Note that Memory is different from Trajectory in that the Memory is explicitly passed to the LLM for usage and the Trajectory contains more information for us to debug.

The `Memory` that we are going to build uses the `messages` structure for tracking conversations:

```json
[
    {
        "role": "user",
        "content": "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."
    },
    {
        "role": "assistant",
        "content": "Hi Maarten and Jay! It's wonderful to meet you."
    }
]
```

This module is rather straightforward and appends new messages each time the user makes a query or when the LLM gives back a reply. As such, the `Memory` module only requires a few lines of code:

In [ ]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str):
        """Add a message to memory."""
        self.messages.append({"role": role, "content": content})

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

We can annotate this module and explore each function in more detail:

In [8]:
from illustrated_agents.chapters.ch4 import memory_annotated; memory_annotated

## 4 - Updating `agent.py`

This added to the `TinyAgent`, which also requires updating a `_step` to track the conversation history following three steps:

1. The user's query is added to the `Memory` module
2. Based on the current memory, the LLM generates a response.
3. The response of the LLM is added to the `Memory` module.

In [13]:
from illustrated_agents.chapters.ch2 import Trajectory

class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory):
        self.llm = llm
        self.memory = memory
        self.tools = None  # Chapter 5: Add Tools
        self.planner = None  # Chapter 6: Add Planning
        self.skills = None  # Chapter 6: Add Skills

        self.trajectory = Trajectory()

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        return self._step()

    def _step(self) -> str:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response.content)
        self.trajectory.add(response)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        # Placeholder - will be implemented in later chapters
        return f"Executed action: {action}"

Let's highlight these steps:

In [14]:
from illustrated_agents.chapters.ch4 import tinyagent_annotated; tinyagent_annotated

Here is a nicer overview of the changes that we made to `agent.py` (red is removed and green is added code):

In [15]:
from illustrated_agents.chapters.ch4 import tinyagents_diff; tinyagents_diff

Next, let's create our `TinyAgent` with `Memory`:

In [16]:
# Add memory to the Agent
memory = Memory()
agent_with_memory = TinyAgent(llm=llm, memory=memory)

We can start filling up the memory by conversing with the model as we did before:

In [17]:
response = agent_with_memory.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

Hi Maarten and Jay! It's great to "meet" you.

**"An Illustrated Guide to AI Agents"** sounds like a fascinating and very timely topic. The field of AI agents is exploding right now, so a guide that is both **informative** and **visual** must be incredibly valuable.

How can I help you today? Are you:

* **Looking for feedback** on the book's content or structure?
* **Drafting marketing copy** or "About the Authors" blurbs?
* **Needing examples** of AI agent concepts for illustration?
* **Just introducing yourselves** in the conversation?

Let me know what's on your mind!


Now that we have memory, we can ask a follow-up question to the original conversation and see if it remember our names correctly.

In [18]:
response = agent_with_memory.run("Hi! What are our names?")
print(response)

Your names are **Maarten and Jay**.


It does! The method by which it does so is filling up the conversation history. Let's see what the current state is.

In [19]:
agent_with_memory.memory.get_messages()

[{'role': 'user',
  'content': "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."},
 {'role': 'assistant',
  'content': 'Hi Maarten and Jay! It\'s great to "meet" you.\n\n**"An Illustrated Guide to AI Agents"** sounds like a fascinating and very timely topic. The field of AI agents is exploding right now, so a guide that is both **informative** and **visual** must be incredibly valuable.\n\nHow can I help you today? Are you:\n\n* **Looking for feedback** on the book\'s content or structure?\n* **Drafting marketing copy** or "About the Authors" blurbs?\n* **Needing examples** of AI agent concepts for illustration?\n* **Just introducing yourselves** in the conversation?\n\nLet me know what\'s on your mind!'},
 {'role': 'user', 'content': 'Hi! What are our names?'},
 {'role': 'assistant', 'content': 'Your names are **Maarten and Jay**.'}]

Note that this entire list is given to the LLM whenever we ask it a new question. That way, it "remembers" the conversation we had before. We say "remembers" because even though it may look like it, it actually has no internal memory. We merely tell the model what the conversation was!

The trajectory itself remains similar since the Trajectory is merely tracking what the model did each step, we can use the Memory to see what the model has actually seen.

In [21]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent_with_memory.trajectory)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [13]:
from illustrated_agents.chapters.ch4 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py  ← Updated (Integrated `Memory` into your `TinyAgent`.)                                            │
│ ├── llm.py                                                                                                      │
│ └── memory.py ← New (Track conversation history.)                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯